<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Tool Schema and Structured Arguments </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Demo: Tool Schema and Structured Arguments

**Purpose:**  
This demo illustrates how the quality of tool schema design significantly impacts agent behavior and reliability.

**Key Topics Covered:**
- Tool schema design
- Structured tool arguments
- Good vs Bad tool descriptions
- Impact on agent decision making

We compare three versions of a book search tool:
1. Bad Tool (poorly designed)
2. Good Tool (well-structured @tool)
3. Pydantic Structured Tool (most robust approach)

**Real Data Source:** All tools scrape live data from https://books.toscrape.com

## 1. Setup

In [1]:
# Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.7/362.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/

In [ ]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

## 2. Bad Tool Design (Poor Schema)

Let's first create a poorly designed tool with vague descriptions and unstructured arguments.

**Issues in this tool:**
- Vague docstring
- No clear argument types or descriptions
- Hard for the agent to understand how to use it correctly

In [ ]:
#  Bad Tool - Poor Design
@tool
def search_books_bad(query: str) -> str:
    """Search for books. This tool has very poor documentation."""
    try:
        url = "https://books.toscrape.com"
        headers = {"User-Agent": "Mozilla/5.0 (compatible; AI-Agent-Demo/1.0)"}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        books = []
        for article in soup.find_all("article", class_="product_pod")[:5]:
            title = article.h3.a.get("title", "No title")
            price = article.find("p", class_="price_color").get_text(strip=True)
            books.append(f"{title} - {price}")

        return f"[BAD TOOL] Searched for '{query}' and found:\n" + "\n".join(books)
    except Exception as e:
        return f"[BAD TOOL] Error: {str(e)}"

## 3. Good Tool Design

This is a well-designed tool with:
- Clear type hints
- Detailed docstring explaining each argument
- Default values
- Return type specification
- Examples in the description

Good tool schemas help agents understand exactly how to call the tool.

In [ ]:
# Good Tool - Well-designed @tool
@tool
def search_books_good(query: str, max_results: int = 5) -> str:
    """Search for popular books from books.toscrape.com.

    Args:
        query (str): General search description (used for display)
        max_results (int): Number of books to return (default 5)

    Returns:
        List of real book titles and prices scraped from https://books.toscrape.com
    """
    try:
        url = "https://books.toscrape.com"
        headers = {"User-Agent": "Mozilla/5.0 (compatible; AI-Agent-Demo/1.0)"}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        books = []
        for article in soup.find_all("article", class_="product_pod")[:max_results]:
            title = article.h3.a.get("title", "No title")
            price = article.find("p", class_="price_color").get_text(strip=True)
            books.append(f"{title} - {price}")

        return f"[GOOD TOOL] Found {len(books)} popular books from https://books.toscrape.com:\n" + "\n".join(books)
    except Exception as e:
        return f"[GOOD TOOL] Error: {str(e)}"

## 4. Pydantic-Based Structured Tool

**Why use Pydantic?**
- Strict schema validation at runtime
- Automatic error messages for invalid inputs
- Better argument parsing by the agent
- Ideal for complex tools with many required fields
- Prevents many common agent failures

In [25]:
# Pydantic Structured Tool
class BookSearchInput(BaseModel):
    query: str = Field(..., description="General description of what kind of books to find")
    max_results: int = Field(5, description="Number of books to return", ge=1, le=10)

def search_books_pydantic(query: str, max_results: int = 5) -> str:
    """Core logic for Pydantic structured tool."""
    try:
        url = "https://books.toscrape.com"
        headers = {"User-Agent": "Mozilla/5.0 (compatible; AI-Agent-Demo/1.0)"}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        books = []
        for article in soup.find_all("article", class_="product_pod")[:max_results]:
            title = article.h3.a.get("title", "No title")
            price = article.find("p", class_="price_color").get_text(strip=True)
            books.append(f"{title} - {price}")

        return f"[PYDANTIC TOOL] Found {len(books)} popular books from https://books.toscrape.com for '{query}':\n" + "\n".join(books)
    except Exception as e:
        return f"[PYDANTIC TOOL] Error: {str(e)}"

search_books_pydantic_tool = StructuredTool.from_function(
    func=search_books_pydantic,
    name="search_books",
    description="Search for popular books from books.toscrape.com with strict Pydantic validation",
    args_schema=BookSearchInput
)

## 5. Creating Agents for Comparison

We now create three separate agents, each using a different version of the tool.
This allows us to clearly observe the impact of tool schema quality on agent behavior.

In [26]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain import hub
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# prompt for real scraping tools
enhanced_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Important Rules for Tool Calling:
- Always use the exact tool name from the available tools.
- For Action Input, provide clean and structured input.
- When a tool has multiple parameters, output them as a valid JSON object.
- Example: {{"query": "popular books", "max_results": 5}}
- Do NOT add extra text like "url =", do NOT wrap the entire input in quotes.

Use the following format exactly:

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action as clean JSON when multiple parameters
Observation: the result of the action
... (repeat if needed)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{agent_scratchpad}"""

enhanced_prompt = PromptTemplate.from_template(enhanced_prompt_template)

# === Create Agents for Comparison ===
bad_executor = AgentExecutor(
    agent=create_react_agent(llm=llm, tools=[search_books_bad], prompt=enhanced_prompt),
    tools=[search_books_bad],
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

good_executor = AgentExecutor(
    agent=create_react_agent(llm=llm, tools=[search_books_good], prompt=enhanced_prompt),
    tools=[search_books_good],
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

pydantic_executor = AgentExecutor(
    agent=create_react_agent(llm=llm, tools=[search_books_pydantic_tool], prompt=enhanced_prompt),
    tools=[search_books_pydantic_tool],
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print(" All three agents created successfully")

All three agents created with enhanced prompt for better argument parsing


## 6. Running the Test  Query on All Three Agents


**Query:** "Find the top 5 popular books with their prices."

We will run this same query on all three tools and compare the traces.

In [28]:
query = "Find the top 5 popular books with their prices."

print("=== 1. BAD TOOL AGENT ===")
bad_response = bad_executor.invoke({"input": query})

print("\n=== 2. GOOD TOOL AGENT ===")
good_response = good_executor.invoke({"input": query})

print("\n=== 3. PYDANTIC STRUCTURED TOOL AGENT ===")
pydantic_response = pydantic_executor.invoke({"input": query})

=== 1. BAD TOOL AGENT ===


> Entering new AgentExecutor chain...
Thought: you should always think about what to do
Action: search_books_bad
Action Input: {"query": "top 5 popular books with prices"}[BAD TOOL] Searched for '{"query": "top 5 popular books with prices"}' and found:
A Light in the Attic - Â£51.77
Tipping the Velvet - Â£53.74
Soumission - Â£50.10
Sharp Objects - Â£47.82
Sapiens: A Brief History of Humankind - Â£54.23I now know the final answer
Final Answer: The top 5 popular books with their prices are:
1. A Light in the Attic - £51.77
2. Tipping the Velvet - £53.74
3. Soumission - £50.10
4. Sharp Objects - £47.82
5. Sapiens: A Brief History of Humankind - £54.23

> Finished chain.

=== 2. GOOD TOOL AGENT ===


> Entering new AgentExecutor chain...
Thought: I should use the search_books_good tool to find the top 5 popular books with their prices.
Action: search_books_good
Action Input: {"query": "popular books", "max_results": 5}[GOOD TOOL] Found 5 popular books from https

## Observation & Analysis

**1. Bad Tool Agent:**
- The agent sends a JSON-like string as a single query parameter.
- The tool receives messy input and has poor formatting.
- This is typical behavior when tool descriptions are vague.

**2. Good Tool Agent:**
- The agent passes structured parameters more cleanly.
- The tool returns properly formatted results with clear labeling.
- Better docstring leads to better argument understanding.

**3. Pydantic Structured Tool Agent:**
- Shows the most structured approach.
- Even though ReAct has parsing limitations, the intent is clearer.
- In production, Pydantic tools provide runtime validation and better error handling.

**Key Learning:**
Well-designed tool schemas (with clear type hints, detailed docstrings, and structured arguments) help agents:
- Choose the right tool more accurately
- Pass arguments correctly
- Reduce hallucinations and parsing errors